# Lesson 1.4 — The Messages API: System, Messages, and Streaming

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Structure a request with system prompt, messages, and max_tokens
- Read the response object (content blocks, stop_reason, usage)
- Stream responses token-by-token for responsive UX
- Handle multi-turn conversations by appending turns

**You will produce:** A streaming chat helper plus a multi-turn conversation loop (offline-safe).

## Concepts

The **Messages API** (`client.messages.create`) is the core integration point. Key fields:

- `model`, `max_tokens` (required ceiling on output), `messages` (list of `{role, content}`), and an optional top-level `system` string for instructions.
- The response exposes `content` (a list of blocks), `stop_reason` (e.g. `end_turn`, `max_tokens`, `tool_use`), and `usage` (`input_tokens`, `output_tokens`).
- **Streaming** (`client.messages.stream(...)`) yields incremental text so users see output as it is produced - important for latency-sensitive UIs. The SDK wraps a REST/SSE transport.
- A conversation is just an accumulating `messages` list: append the assistant's reply, then the next user turn.

> **Reference:** [Anthropic - Messages API](https://docs.anthropic.com/en/api/messages)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A structured request with a system prompt

Put role/behavior instructions in `system`, task content in `messages`.

In [ ]:
def ask(system, user):
    if not HAVE_API:
        return mock_text(user)
    msg = client.messages.create(
        model=MODEL_SONNET, max_tokens=300,
        system=system,
        messages=[{"role": "user", "content": user}],
    )
    print("stop_reason:", msg.stop_reason, "| usage:", msg.usage)
    return msg.content[0].text

print(ask("You are a terse SRE assistant.", "What is a p99 latency?"))

## Code-Along: Step 2 — Stream the response

Use the streaming context manager to print text as it arrives.

In [ ]:
def stream_answer(user):
    if not HAVE_API:
        print(mock_text(user)); return
    with client.messages.stream(
        model=MODEL_SONNET, max_tokens=300,
        messages=[{"role": "user", "content": user}],
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
    print()

stream_answer("List three REST best practices.")

## Code-Along: Step 3 — Multi-turn conversation

Append each assistant reply before the next user turn.

In [ ]:
history = [{"role": "user", "content": "My name is Dana."}]
if HAVE_API:
    r1 = client.messages.create(model=MODEL_HAIKU, max_tokens=100, messages=history)
    history.append({"role": "assistant", "content": r1.content[0].text})
else:
    history.append({"role": "assistant", "content": "[MOCK] Hi Dana!"})
history.append({"role": "user", "content": "What's my name?"})
print(ask("", "") if False else history[-1])

## Exercise

Build a `Chat` class holding a `messages` list with a `.send(user_text)` method that calls the API (or mock), appends both turns, and returns the reply. Keep it offline-safe.

In [ ]:
class Chat:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
    def send(self, user_text):
        # TODO: append the user turn
        # TODO: call client.messages.create (or mock) and read content[0].text
        # TODO: append the assistant turn and return the reply
        pass

## Key Takeaways

- messages.create needs model, max_tokens, and messages; system holds instructions.
- Inspect stop_reason and usage on every response for control and cost tracking.
- Streaming improves perceived latency by emitting text incrementally.
- Multi-turn chat is an accumulating messages list - append replies then the next question.

---

**Next:** [Lesson 1.5 — Vision Input and Prompt Caching](./lesson-1.5-vision-and-caching.ipynb)